# Triaxial Compression — Reference State & Profile Evolution

Analysis for `triaxial_compression.lmp`.  Two parts:

1. **Reference state** (ε = 0, piston held at v = 0 over the 150k pre-compression window): averaged profiles **with confidence intervals** for the total stress σ_zzᵗ(z), the solvent-only stress σ_s,ss(z), the solvent mass density ρ_s(z), and the solvent mass fraction ρ_s/ρ_{s,0}(z).
2. **Profile evolution vs. timestep** for each of those observables, sampled as 10 curves starting **after** the slab reaches its 10 % compression (the relaxation phase).

Colorblind-friendly palette; gel interior shaded; in/out-gel means annotated.

## Before running: files to copy from the cluster

Copy the run's output from the cluster into your local `flow_data_local` tree.
Set `RUN_ID` and `sim_name` in the **Config** cell to match the run, then place
the files as below (the notebook builds every path from those two strings).

**Into** `flow_data_local/compression/<RUN_ID>/`  *(cluster:* `.../output_files/`*)*

| file pattern | cluster subfolder |
|---|---|
| `sigmazz_polymer_<sim_name>.dat`, `sigmazz_solvent_<sim_name>.dat` | `stress_data/` |
| `sigmazz_polymer_ref_<sim_name>.dat`, `sigmazz_solvent_ref_<sim_name>.dat` | `stress_data/` |
| `strain_zz_<sim_name>.dat` | `stress_data/` |
| `solvent_density_z_<sim_name>.dat`, `solvent_density_z_ref_<sim_name>.dat` | `chemical_potential/` |
| `piston_position_<sim_name>.dat` | `piston_data/` |
| `pairs_<sim_name>.dump`, `pairs_ref_<sim_name>.dump` | `pair_data/` |
| `polymer_pairs_<sim_name>.dump`, `polymer_pairs_ref_<sim_name>.dump` | `pair_data/` |
| `bonds_<sim_name>.dump`, `bonds_ref_<sim_name>.dump` | `pair_data/` |

**Into** `flow_data_local/traj_files.nosync/`  *(cluster:* `.../traj_files/`*)*

| file pattern | cluster subfolder |
|---|---|
| `traj_stress_<sim_name>.lammpstrj` | `traj_files/` |
| `traj_ref_<sim_name>.lammpstrj` | `traj_files/` |

Plots are written to `flow_data_local/plots/compression/<RUN_ID>/` (created automatically).

*Optional:* the run also writes high-resolution `sigmazz_*_fine_support/piston_*` and
`solvent_density_z_fine_*` files (support/piston windows). This notebook plots the
coarse profiles only; copy the `*_fine_*` files too if you later add high-res panels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from scipy import stats
from scipy.interpolate import interp1d
from pathlib import Path

# ---- user rcParams (matches compression_analysis.ipynb) ----
plt.rcParams.update({
    'font.family': 'CMU Serif',
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'CMU Serif',
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 25,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
    'legend.fontsize': 23,
    'figure.titlesize': 22,
    'axes.unicode_minus': False,
})

# ---- colorblind-friendly palettes ----
# Wong (2011) categorical palette for the reference single-curve plots.
WONG = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73','vermillion':'#D55E00',
        'skyblue':'#56B4E9','yellow':'#F0E442','reddishpurple':'#CC79A7','black':'#000000'}
# 'cividis' is the most CVD-safe sequential map -> used for the time gradient.
EVO_CMAP = 'cividis'
GEL_SHADE = dict(color='0.6', alpha=0.15, zorder=0)   # neutral grey, CVD-safe

print('Imports + style ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG — only change the lines in this block to switch datasets
# ══════════════════════════════════════════════════════════════════════════
# sim_name is the exact suffix LAMMPS appends to every output file:
#     <DATANAME>_<INTERACTION>_<NSTEPS>
DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000000"
INTERACTION = "1.0_1.0"                 # epsSS_epsSP for the triaxial run
NSTEPS      = 4000000                   # triaxial production steps; None -> auto-detect from files
RUN_ID      = "periodic_rho04_1.4M_4M_SWEEP1"  # local folder label under flow_data_local/{compression,plots}

# ──────────────────────────────────────────────────────────────────────────
#  COMPRESSION-SWEEP STRAIN LEVELS  ← the one place that defines the sweep
# ──────────────────────────────────────────────────────────────────────────
# These are the cumulative strain targets and MUST match COMPRESSIONS=(...) in
# triaxial_compression.batch on the cluster.  triaxial_compression.lmp tags
# EVERY production file with _c<level> — even a single-target run writes its one
# default level (e.g. _c0.10) — so this list is how the notebook finds files.
# There is NO such thing as an un-tagged production run; a single run is just a
# one-element list here.
#   • The sweep-summary plot (§6, last cell) loops over ALL levels below.
#   • The detailed per-level plots (§1–§5) use the ONE level picked by DETAIL_IDX.
COMP_LEVELS = ["0.10", "0.20", "0.30", "0.40"]   # ← edit to match COMPRESSIONS=(...) in the batch
DETAIL_IDX  = -1     # index into COMP_LEVELS for the detailed §1–§5 plots (-1 = last/most-compressed)
# ══════════════════════════════════════════════════════════════════════════

assert COMP_LEVELS, "COMP_LEVELS is empty — list at least one strain level (e.g. [\"0.10\"])."
COMP_LEVEL = COMP_LEVELS[DETAIL_IDX]     # single level used by the per-level (§1–§5) plots
print(f'>>> SWEEP has {len(COMP_LEVELS)} strain level(s): {COMP_LEVELS}')
print(f'>>> detailed plots (§1–§5) use level [{DETAIL_IDX}] = {COMP_LEVEL}')

DATA_DIR  = Path("../../flow_data_local/compression") / RUN_ID
PLOT_DIR  = Path("../../flow_data_local/plots/compression") / RUN_ID
TRAJ_DIR  = Path("../../flow_data_local/traj_files.nosync")
# auto-create the local folders for this run (idempotent; safe to re-run)
for _d in (DATA_DIR, PLOT_DIR, TRAJ_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print('folders ready:', DATA_DIR, '|', PLOT_DIR, '|', TRAJ_DIR)

# Build sim_name.  If NSTEPS is None, auto-detect it from any stress file
# already in DATA_DIR (e.g. after the Expanse sync); else use it directly.
if NSTEPS is None:
    import re as _re
    _cands = sorted(DATA_DIR.glob(f'sigmazz_polymer_{DATANAME}_{INTERACTION}_*.dat'))
    if not _cands:
        raise ValueError('NSTEPS is None and no sigmazz_polymer_*.dat in DATA_DIR yet '
                         '-- set NSTEPS explicitly, or run the Expanse sync cell first.')
    NSTEPS = int(_re.match(rf'.*_{_re.escape(INTERACTION)}_(\d+)\.dat$', _cands[-1].name).group(1))
    print(f'auto-detected NSTEPS = {NSTEPS} from {_cands[-1].name}')
sim_name = f'{DATANAME}_{INTERACTION}_{NSTEPS}'    # base tag (NO _c); used verbatim by every *_ref file
LEVEL_TAG = f'_c{COMP_LEVEL}'                       # per-level suffix on PRODUCTION files (always set; .lmp always tags _c)

# ---- analysis parameters ----
binWidth      = 2.0      # coarse z-bin (sigma); must match triaxial_compression.lmp
binWidth_fine = 0.5      # fine z-bin (sigma)        "
# kinetic stress removed per request -- all recorded stresses are virial-only
solvent_mass  = 1.0      # solvent bead mass (LJ); rho_mass = solvent_mass * rho_number
comp_percent  = 0.1      # compression strain threshold (halt); evolution starts after this
n_curves      = 10       # target time-evolution curves (matches num_stress_curves)
ci_level      = 0.95
gel_thresh    = 0.05     # gel interior = bins where |sigma_p,zz| > gel_thresh*max
flat_tol      = 0.15     # final evolution curve "flat inside gel" if rel. spread < this
wall_margin   = 4.0      # sigma trimmed off support/piston ends before flat test + mean (skip wall-depletion layers)
BOND_SIGN     = +1.0     # flip to -1 if sigma_p,pp comes out opposite the group sigma_p

# ---- file paths ----
def _tag(nm): return '' if nm.endswith('_ref') else LEVEL_TAG   # *_ref files are shared across levels (no _c)
def D(name):  return DATA_DIR / f'{name}_{sim_name}{_tag(name)}.dat'
def Ddump(name): return DATA_DIR / f'{name}_{sim_name}{_tag(name)}.dump'
def T(name):  return TRAJ_DIR / f'{name}_{sim_name}{_tag(name)}.lammpstrj'

# stress (coarse): sigma_zz polymer/solvent, production + reference
F_SZZ_P      = D('sigmazz_polymer');      F_SZZ_S      = D('sigmazz_solvent')
F_SZZ_P_REF  = D('sigmazz_polymer_ref');  F_SZZ_S_REF  = D('sigmazz_solvent_ref')
# solvent density (coarse), production + reference  (cols: ... density/number density/mass)
F_DENS       = DATA_DIR / f'solvent_density_z_{sim_name}{LEVEL_TAG}.dat'
F_DENS_REF   = DATA_DIR / f'solvent_density_z_ref_{sim_name}.dat'
# pair/bond dumps for ss and pp stress
F_PAIRS      = Ddump('pairs');            F_PAIRS_REF      = Ddump('pairs_ref')
F_PPAIRS     = Ddump('polymer_pairs');    F_PPAIRS_REF     = Ddump('polymer_pairs_ref')
F_BONDS      = Ddump('bonds');            F_BONDS_REF      = Ddump('bonds_ref')
F_TRAJSTRESS = T('traj_stress');          F_TRAJREF        = T('traj_ref')
# strain + piston position (to find the compression-halt timestep)
F_STRAIN     = D('strain_zz')
F_PISTON_POS = DATA_DIR / f'piston_position_{sim_name}{LEVEL_TAG}.dat'
F_PISTON_FORCE = DATA_DIR / f'piston_force_{sim_name}{LEVEL_TAG}.dat'
F_PISTON_FORCE_AVG = DATA_DIR / f'piston_force_avg_{sim_name}{LEVEL_TAG}.dat'  # LMP block-averaged
print('Config set for', sim_name, '(LEVEL_TAG=' + (LEVEL_TAG or 'none') + ')')

## Sync data from Expanse (only when needed)


In [ ]:
# === Sync triaxial-compression data from Expanse (only when needed) ===
# Equivalent to the volume_of_mixing sync cell, adapted to this notebook's files.
# Pulls exactly what the loader below reads for the current sim_name:
#   • core .dat : sigmazz_(polymer|solvent)[_ref], solvent_density_z[_ref],
#                 strain_zz, piston_position, piston_force       -> DATA_DIR
#   • pair/bond : pairs[_ref], polymer_pairs[_ref], bonds[_ref] (.dump) -> DATA_DIR
#   • trajs     : traj_stress, traj_ref (.lammpstrj)            -> TRAJ_DIR
# An Expanse login happens ONLY if a REQUIRED core file is missing locally.
# Set FORCE_SYNC = True to also refresh the large pair/bond/traj files when the
# core files are already present.
import paramiko, getpass, stat
from pathlib import Path

EXPANSE_HOST = "login.expanse.sdsc.edu"
EXPANSE_USER = "dpollard"
RUNS_ROOT    = "/home/dpollard/Documents/lammps_runs/triaxial_compression"   # working dirs (output_files/*)
TRAJ_ROOT    = "/expanse/lustre/scratch/dpollard/temp_project/lammps_trajectories"  # .lammpstrj on scratch
STAGE_DIR    = f"{RUNS_ROOT}/triaxial_stage"
FORCE_SYNC   = False     # True -> sync even if core files already present (refresh dumps/trajs)

# Files the loader reads (built from the F_* paths defined in the Config cell),
# split by local destination.  REQUIRED = the core files needed for the main plots.
# Files span EVERY level in COMP_LEVELS (the sweep-summary and combined-sweep
# plots need per-level data), plus the shared _ref files pulled once.  Missing
# box_dimensions here is what made the stress-strain sweep plot come up empty.
_PROD_DAT  = ('sigmazz_polymer', 'sigmazz_solvent', 'solvent_density_z',
              'strain_zz', 'piston_position', 'piston_force', 'piston_force_avg',
              'box_dimensions')
_PROD_DUMP = ('pairs', 'polymer_pairs', 'bonds')
def _level_data_files(lvl):
    b = f'{sim_name}_c{lvl}'
    return ([DATA_DIR / f'{n}_{b}.dat'  for n in _PROD_DAT]
          + [DATA_DIR / f'{n}_{b}.dump' for n in _PROD_DUMP])
def _level_traj_files(lvl):
    return [TRAJ_DIR / f'traj_stress_{sim_name}_c{lvl}.lammpstrj']

_REF_DATA = [F_SZZ_P_REF, F_SZZ_S_REF, F_DENS_REF, F_PAIRS_REF, F_PPAIRS_REF, F_BONDS_REF]
DATA_FILES = list(_REF_DATA)
TRAJ_FILES = [F_TRAJREF]
REQUIRED   = [F_SZZ_P_REF, F_SZZ_S_REF, F_DENS_REF]
for _lvl in COMP_LEVELS:
    DATA_FILES += _level_data_files(_lvl)
    TRAJ_FILES += _level_traj_files(_lvl)
    _b = f'{sim_name}_c{_lvl}'
    REQUIRED   += [DATA_DIR / f'sigmazz_polymer_{_b}.dat',
                   DATA_DIR / f'sigmazz_solvent_{_b}.dat',
                   DATA_DIR / f'solvent_density_z_{_b}.dat',
                   DATA_DIR / f'strain_zz_{_b}.dat',
                   DATA_DIR / f'piston_force_{_b}.dat',
                   DATA_DIR / f'box_dimensions_{_b}.dat']

missing_req = [f for f in REQUIRED               if not Path(f).exists()]
missing_all = [f for f in (DATA_FILES+TRAJ_FILES) if not Path(f).exists()]
if not FORCE_SYNC and not missing_req:
    print(f"All {len(REQUIRED)} required files present locally "
          f"({len(missing_all)} optional pair/traj file(s) missing) — skipping Expanse login.")
else:
    why = "FORCE_SYNC" if (FORCE_SYNC and not missing_req) else f"{len(missing_req)} required file(s) missing"
    print(f"Syncing from Expanse ({why}); {len(missing_all)} of "
          f"{len(DATA_FILES)+len(TRAJ_FILES)} target files missing locally.")

    def _bash_list(paths):
        return " ".join(f'"{Path(p).name}"' for p in paths)
    DATA_BN, TRAJ_BN = _bash_list(DATA_FILES), _bash_list(TRAJ_FILES)

    # Stage the newest match for each wanted basename with ONE find per tree
    # (basename->newest index; cp -p preserves mtimes so the SFTP skip fires on
    # reruns).  .dat/.dump come from the runs tree, .lammpstrj from scratch.
    stage_script = r"""
set -u
RUNS="__RUNS__"; TRAJ="__TRAJ__"; STAGE="__STAGE__"
rm -rf "$STAGE"; mkdir -p "$STAGE/data" "$STAGE/traj"
declare -A NEWEST_D
while IFS= read -r line; do p=${line#* }; b=${p##*/}
  [ -z "${NEWEST_D[$b]:-}" ] && NEWEST_D[$b]="$p"
done < <(find "$RUNS" \( -name '*.dat' -o -name '*.dump' \) -not -path '*/triaxial_stage/*' -printf '%T@ %p\n' 2>/dev/null | sort -rn)
for B in __DATA_BN__; do S="${NEWEST_D[$B]:-}"; [ -n "$S" ] && cp -p "$S" "$STAGE/data/" 2>/dev/null || true; done
declare -A NEWEST_T
while IFS= read -r line; do p=${line#* }; b=${p##*/}
  [ -z "${NEWEST_T[$b]:-}" ] && NEWEST_T[$b]="$p"
done < <(find "$TRAJ" "$RUNS" -name '*.lammpstrj' -not -path '*/triaxial_stage/*' -printf '%T@ %p\n' 2>/dev/null | sort -rn)
for B in __TRAJ_BN__; do S="${NEWEST_T[$B]:-}"; [ -n "$S" ] && cp -p "$S" "$STAGE/traj/" 2>/dev/null || true; done
echo "  staged: $(ls "$STAGE/data" 2>/dev/null | wc -l) data, $(ls "$STAGE/traj" 2>/dev/null | wc -l) traj"
"""
    stage_script = (stage_script.replace("__RUNS__", RUNS_ROOT).replace("__TRAJ__", TRAJ_ROOT)
                    .replace("__STAGE__", STAGE_DIR).replace("__DATA_BN__", DATA_BN)
                    .replace("__TRAJ_BN__", TRAJ_BN))

    password = getpass.getpass(f"Expanse password for {EXPANSE_USER}: ")
    totp     = getpass.getpass("TOTP / verification code: ")
    def auth_handler(title, instructions, prompt_list):
        return [password if "password" in p.strip().lower() else totp for p, _ in prompt_list]

    print("Connecting to Expanse...")
    transport = paramiko.Transport((EXPANSE_HOST, 22))
    transport.connect()
    transport.auth_interactive(EXPANSE_USER, auth_handler)
    ssh = paramiko.SSHClient(); ssh._transport = transport

    print("Step 1 — staging files on Expanse...")
    _, stdout, stderr = ssh.exec_command("bash -s", get_pty=False)
    stdout.channel.sendall(stage_script.encode()); stdout.channel.shutdown_write()
    print(stdout.read().decode())

    print("Step 2 — downloading via SFTP (skips files already present)...")
    sftp = ssh.open_sftp()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    TRAJ_DIR.mkdir(parents=True, exist_ok=True)
    def sftp_pull(remote_dir, local_dir):
        local_dir = Path(local_dir); local_dir.mkdir(parents=True, exist_ok=True)
        try: entries = sftp.listdir_attr(remote_dir)
        except FileNotFoundError: return
        for e in entries:
            rp, lp = f"{remote_dir}/{e.filename}", local_dir / e.filename
            if stat.S_ISDIR(e.st_mode):
                sftp_pull(rp, lp); continue
            if lp.exists() and lp.stat().st_mtime >= e.st_mtime:
                continue
            sftp.get(rp, str(lp))
    sftp_pull(f"{STAGE_DIR}/data", DATA_DIR)
    sftp_pull(f"{STAGE_DIR}/traj", TRAJ_DIR)
    sftp.close(); ssh.close()
    print("Sync complete.")


In [ ]:
# ============================ HELPER FUNCTIONS =============================
def read_print_file(filepath, col_names=None):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    if col_names is None: col_names = [f'col_{i}' for i in range(arr.shape[1])]
    return {name: arr[:, i] for i, name in enumerate(col_names)}

def read_ave_time_file(filepath):
    """fix ave/time mode vector -> list of (timestep, bin_idx, values)."""
    out = []
    with open(filepath) as f:
        lines = [l for l in f if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) == 2:
            ts, nrows = int(parts[0]), int(parts[1])
            vals = []
            for j in range(1, nrows + 1):
                if i + j < len(lines):
                    vp = lines[i + j].split()
                    if len(vp) == 2: vals.append(float(vp[1]))
            if vals: out.append((ts, np.arange(1, len(vals)+1), np.array(vals)))
            i += nrows + 1
        else:
            i += 1
    return out

def read_ave_chunk_file(filepath):
    """fix ave/chunk -> list of (timestep, array[rows, cols]).
    cols: [chunk_id, Coord1, Ncount, val1(, val2...)]."""
    snaps = []
    with open(filepath) as f:
        lines = [l for l in f if l.strip() and not l.startswith('#')]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) in (2, 3):
            try: ts, nch = int(parts[0]), int(parts[1])
            except ValueError:
                i += 1; continue
            rows = []
            for j in range(1, nch + 1):
                if i + j < len(lines): rows.append([float(v) for v in lines[i + j].split()])
            if rows: snaps.append((ts, np.array(rows)))
            i += nch + 1
        else:
            i += 1
    return snaps

def read_strain_file(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    # cols: step  L_initial  L_current  -> eps = (L0 - L)/L0
    ts = arr[:, 0].astype(int); L0 = arr[:, 1]; L = arr[:, 2]
    eps = (L0 - L) / L0
    return ts, eps

def mean_ci(stack, ci=0.95):
    """stack: (n_samples, n_bins) -> (mean, lo, hi) per bin via t-interval."""
    stack = np.asarray(stack, float)
    n = stack.shape[0]
    m = np.nanmean(stack, axis=0)
    if n < 2:
        return m, m, m
    se = stats.sem(stack, axis=0, nan_policy='omit')
    half = se * stats.t.ppf(0.5 + ci/2, df=n-1)
    return m, m - half, m + half

def read_pairs_local_dump(filepath):
    """dump local (pair/local) -> list (timestep, box, data[n,7])
    data cols: id1 id2 type1 type2 fx fy fz."""
    frames = []
    with open(filepath) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'ITEM: TIMESTEP':
            ts = int(lines[i+1]); n = int(lines[i+3])
            xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
            zb = list(map(float, lines[i+7].split()))
            box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
            s = i + 9
            rows = [[float(v) for v in lines[s+j].split()] for j in range(n) if s+j < len(lines)]
            frames.append((ts, box, np.array(rows) if rows else np.zeros((0, 7))))
            i = s + n
        else:
            i += 1
    return frames

def read_bonds_dump(filepath):
    """dump local (bond/local) -> list (timestep, box, data[n,5])
    data cols: batom1 batom2 btype force dist."""
    frames = []
    with open(filepath) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'ITEM: TIMESTEP':
            ts = int(lines[i+1]); n = int(lines[i+3])
            xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
            zb = list(map(float, lines[i+7].split()))
            box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
            s = i + 9
            rows = [[float(v) for v in lines[s+j].split()] for j in range(n) if s+j < len(lines)]
            frames.append((ts, box, np.array(rows) if rows else np.zeros((0, 5))))
            i = s + n
        else:
            i += 1
    return frames

def _stream_traj_positions(traj_file, target_ts, types_keep):
    """Return {ts: (box, {id:(x,y,z)})} for atoms whose type is in types_keep."""
    out = {}
    with open(traj_file) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if 'ITEM: TIMESTEP' in lines[i]:
            ts = int(lines[i+1]); n = int(lines[i+3])
            if ts in target_ts:
                xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
                zb = list(map(float, lines[i+7].split()))
                box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
                pos = {}
                for j in range(n):
                    p = lines[i+9+j].split()
                    if int(p[1]) in types_keep:
                        pos[int(p[0])] = (float(p[3]), float(p[4]), float(p[5]))
                out[ts] = (box, pos)
            i += 9 + n
        else:
            i += 1
    return out

print('Readers defined')

In [ ]:
# ===== solvent-only (ss) and polymer-only (pp) stress from pair/bond dumps =====
def _virial_zz_from_pairs(pair_data, pos, box, binWidth, type_filter):
    """sigma_zz(z) from pair/local forces, both atoms in type_filter (a set).
    Returns (z_bins, sigma_zz) -- virial only, no kinetic term."""
    lx = box['x'][1]-box['x'][0]; ly = box['y'][1]-box['y'][0]; lz = box['z'][1]-box['z'][0]
    zlo = box['z'][0]
    nb = int(round(lz / binWidth)); binvol = lx*ly*binWidth
    z_bins = zlo + (np.arange(nb)+0.5)*binWidth
    szz = np.zeros(nb)
    if len(pair_data):
        m = np.isin(pair_data[:,2].astype(int), list(type_filter)) & \
            np.isin(pair_data[:,3].astype(int), list(type_filter))
        sp = pair_data[m]
        if len(sp):
            id1 = sp[:,0].astype(int); id2 = sp[:,1].astype(int); fz = sp[:,6]
            keep = np.array([(a in pos and b in pos) for a,b in zip(id1,id2)])
            if keep.any():
                id1=id1[keep]; id2=id2[keep]; fz=fz[keep]
                p1 = np.array([pos[a] for a in id1]); p2 = np.array([pos[b] for b in id2])
                dz = p2[:,2]-p1[:,2]; dz -= lz*np.round(dz/lz)
                w = -(dz*fz)
                zmid = p1[:,2] + dz*0.5
                bi = ((zmid - zlo)/binWidth).astype(int)
                ok = (bi>=0)&(bi<nb)
                np.add.at(szz, bi[ok], w[ok])
    return z_bins, szz/binvol

def _bond_virial_zz(bond_data, pos, box, binWidth, bond_sign=1.0):
    """FENE bond sigma_zz(z): W_zz = bond_sign*(force/|r|)*dz^2, midpoint-binned."""
    lx = box['x'][1]-box['x'][0]; ly = box['y'][1]-box['y'][0]; lz = box['z'][1]-box['z'][0]
    zlo = box['z'][0]
    nb = int(round(lz / binWidth)); binvol = lx*ly*binWidth
    szz = np.zeros(nb)
    if len(bond_data):
        b1 = bond_data[:,0].astype(int); b2 = bond_data[:,1].astype(int); force = bond_data[:,3]
        keep = np.array([(a in pos and b in pos) for a,b in zip(b1,b2)])
        if keep.any():
            b1=b1[keep]; b2=b2[keep]; force=force[keep]
            p1=np.array([pos[a] for a in b1]); p2=np.array([pos[b] for b in b2])
            dr = p2-p1
            dr[:,0]-=lx*np.round(dr[:,0]/lx); dr[:,1]-=ly*np.round(dr[:,1]/ly); dr[:,2]-=lz*np.round(dr[:,2]/lz)
            r = np.sqrt((dr**2).sum(1)); r[r==0]=np.nan
            w = bond_sign*(force/r)*dr[:,2]**2
            zmid = p1[:,2] + dr[:,2]*0.5
            bi = ((zmid - zlo)/binWidth).astype(int); ok=(bi>=0)&(bi<nb)&np.isfinite(w)
            np.add.at(szz, bi[ok], w[ok])
    return szz/binvol

def compute_ss_stress(pairs_file, traj_file, binWidth, z_target):
    """Per-frame sigma_s,ss(z), virial only, on z_target. Returns (timesteps, stack[n,nz])."""
    pf = read_pairs_local_dump(pairs_file)
    if not pf: return [], np.zeros((0,len(z_target)))
    tgt = {ts for ts,_,_ in pf}
    traj = _stream_traj_positions(traj_file, tgt, {3})
    ts_out, stack = [], []
    for ts, box, pdata in pf:
        if ts not in traj: continue
        _, pos = traj[ts]
        zb, szz = _virial_zz_from_pairs(pdata, pos, box, binWidth, {3})
        stack.append(np.interp(z_target, zb, szz, left=np.nan, right=np.nan)); ts_out.append(ts)
    return ts_out, np.array(stack)

def compute_pp_stress(ppairs_file, bonds_file, traj_file, binWidth, z_target, bond_sign=1.0):
    """Per-frame sigma_p,pp(z) = pp pair virial + FENE bond virial, virial only, on z_target."""
    pf = read_pairs_local_dump(ppairs_file)
    bf = {ts:(box,data) for ts,box,data in read_bonds_dump(bonds_file)} if Path(bonds_file).exists() else {}
    if not pf: return [], np.zeros((0,len(z_target)))
    tgt = {ts for ts,_,_ in pf}
    traj = _stream_traj_positions(traj_file, tgt, {1,2})
    ts_out, stack = [], []
    for ts, box, pdata in pf:
        if ts not in traj: continue
        _, pos = traj[ts]
        zb, szz = _virial_zz_from_pairs(pdata, pos, box, binWidth, {1,2})
        if ts in bf:
            szz = szz + _bond_virial_zz(bf[ts][1], pos, box, binWidth, bond_sign)
        stack.append(np.interp(z_target, zb, szz, left=np.nan, right=np.nan)); ts_out.append(ts)
    return ts_out, np.array(stack)

print('ss / pp stress reconstruction defined')

## Load data: coordinates, gel bounds, reference & production profiles

In [ ]:
# ---- box z-extent (fixed: only the piston moves) + wall positions ----
def _read_box_z(dumpfile):
    """Return (zlo, zhi) from the first frame of a LAMMPS dump."""
    with open(dumpfile) as f:
        head = [next(f) for _ in range(9)]
    return tuple(map(float, head[7].split()))          # line 7 = z BOX BOUNDS
Z_LO, Z_HI = _read_box_z(F_PAIRS_REF)
LZ = Z_HI - Z_LO
zn = lambda z: (np.asarray(z, float) - Z_LO) / LZ      # fractional box height, ~[0, 1]

def _wall_z_first_frame(traj, types=(4, 5)):
    """Mean z of each atom type in the FIRST frame of a LAMMPS dump trajectory."""
    zc = {t: [] for t in types}
    if not Path(traj).exists():
        return {t: np.nan for t in types}
    with open(traj) as f:
        cols = None
        for line in f:                                  # advance to the ATOMS header
            if line.startswith('ITEM: ATOMS'):
                cols = line.split()[2:]; break
        ti, zi = cols.index('type'), cols.index('z')
        for line in f:
            if line.startswith('ITEM:'): break          # stop at next frame
            p = line.split(); t = int(float(p[ti]))
            if t in zc: zc[t].append(float(p[zi]))
    return {t: (np.mean(v) if v else np.nan) for t, v in zc.items()}
_wz = _wall_z_first_frame(F_TRAJREF, (4, 5))
z_support = _wz.get(4, np.nan)                          # type 4 = frozen support sheet
z_piston  = _wz.get(5, np.nan)                          # type 5 = piston sheet
print(f'box z-extent (fixed): [{Z_LO:.2f}, {Z_HI:.2f}]  Lz = {LZ:.2f}')
print(f'support (type4) z = {z_support:.2f}  |  piston (type5) z = {z_piston:.2f}')

# piston z(t): output only during compression; np.interp clamps to the held
# post-halt value, so any evolution timestep maps to the compressed position.
if Path(F_PISTON_POS).exists():
    _pp = np.loadtxt(F_PISTON_POS, comments='#')
    _pt, _pz = _pp[:, 0], _pp[:, 1]
    def piston_z_at(t): return float(np.interp(t, _pt, _pz))
else:
    def piston_z_at(t): return z_piston
print(f'piston z(t): {_pt[0]:.1f} -> {_pt[-1]:.1f} steps, '
      f'z {_pz[0]:.1f} -> {_pz[-1]:.1f}' if Path(F_PISTON_POS).exists() else 'piston z(t): file missing')

# ---- coarse sigma_zz (production) -> z grid + total/partial time series ----
szz_p = read_ave_time_file(F_SZZ_P)
szz_s = read_ave_time_file(F_SZZ_S)
n_prod = len(szz_p)
prod_ts   = np.array([szz_p[i][0] for i in range(n_prod)])
sig_p_zz  = [szz_p[i][2] for i in range(n_prod)]
sig_s_zz  = [szz_s[i][2] for i in range(n_prod)]
sig_t_zz  = [sig_p_zz[i] + sig_s_zz[i] for i in range(n_prod)]      # total sigma_zz
bins_z    = szz_p[0][1]
z_coords  = Z_LO + bins_z * binWidth - binWidth/2.0   # bin centers in real box coords
print(f'coarse stress: {n_prod} production snapshots, {len(z_coords)} z-bins '
      f'[{z_coords.min():.1f}, {z_coords.max():.1f}]')

# ---- reference coarse sigma_zz (multi-snapshot -> mean + CI) ----
szz_p_ref = read_ave_time_file(F_SZZ_P_REF)
szz_s_ref = read_ave_time_file(F_SZZ_S_REF)
ref_p_stack = np.array([s[2] for s in szz_p_ref])
ref_s_stack = np.array([s[2] for s in szz_s_ref])
ref_t_stack = ref_p_stack + ref_s_stack
print(f'reference stress: {len(szz_p_ref)} snapshots')

# total sigma_zz reference mean/CI
sig_t_ref_m, sig_t_ref_lo, sig_t_ref_hi = mean_ci(ref_t_stack, ci_level)
# group-based polymer reference (for the pp sign-validation cell)
sig_p_ref_m = np.nanmean(ref_p_stack, axis=0)

# ---- gel bounds from reference polymer sigma_zz ----
_pm = np.abs(sig_p_ref_m); _pmax = float(_pm.max())
_gel = (_pm > gel_thresh*_pmax) if _pmax > 0 else np.zeros(len(z_coords), bool)
z_gel_lo = float(z_coords[_gel].min()) if _gel.any() else z_coords[0]
z_gel_hi = float(z_coords[_gel].max()) if _gel.any() else z_coords[-1]
in_gel  = (z_coords >= z_gel_lo) & (z_coords <= z_gel_hi)
print(f'gel interior: z in [{z_gel_lo:.1f}, {z_gel_hi:.1f}]  ({in_gel.sum()} bins)')

# ---- compression-halt timestep (evolution starts AFTER this) ----
strain_ts, strain_eps = read_strain_file(F_STRAIN)
_reached = np.where(strain_eps >= comp_percent)[0]
halt_ts = int(strain_ts[_reached[0]]) if len(_reached) else int(strain_ts[-1])
print(f'compression halt at step {halt_ts} (eps>={comp_percent}); '
      f'{"reached" if len(_reached) else "NOT reached - using last step"}')


In [ ]:
# ---- solvent density (production + reference): cols chunk,Coord1,Ncount,n_dens,m_dens ----
def _load_density(path):
    snaps = read_ave_chunk_file(path)
    ts = np.array([s[0] for s in snaps])
    z  = snaps[0][1][:, 1]
    nden = np.array([s[1][:, 3] for s in snaps])        # density/number
    mden = np.array([s[1][:, 4] for s in snaps])        # density/mass
    return ts, z, nden, mden

dens_ts, dens_z, dens_n, dens_m = _load_density(F_DENS)
rdens_ts, rdens_z, rdens_n, rdens_m = _load_density(F_DENS_REF)
print(f'density: {len(dens_ts)} production, {len(rdens_ts)} reference snapshots on {len(dens_z)} bins')

# reference mass density mean/CI
rho_ref_m, rho_ref_lo, rho_ref_hi = mean_ci(rdens_m, ci_level)
# bulk reference solvent mass density rho_{s,0} = mean over reservoir bins (high-density)
_rmax = float(np.nanmax(rho_ref_m)); _res = rho_ref_m >= 0.85*_rmax
rho_s0 = float(np.nanmean(rho_ref_m[_res]))
print(f'rho_s,0 (bulk reservoir reference mass density) = {rho_s0:.4f}')

# reference mass fraction stack = each ref snapshot / rho_s0
mf_ref_stack = rdens_m / rho_s0
mf_ref_m, mf_ref_lo, mf_ref_hi = mean_ci(mf_ref_stack, ci_level)

In [ ]:
# ---- solvent-only (ss) and polymer-only (pp) stress: reference + production ----
# Reference: multi-frame *_ref dumps -> mean + CI.  Production: filter post-halt.
ss_ref_m = ss_ref_lo = ss_ref_hi = None
ss_prod_ts = None; ss_prod_stack = None
if F_PAIRS_REF.exists() and F_TRAJREF.exists():
    _ts, _stk = compute_ss_stress(F_PAIRS_REF, F_TRAJREF, binWidth, z_coords)
    if len(_stk): ss_ref_m, ss_ref_lo, ss_ref_hi = mean_ci(_stk, ci_level)
    print(f'sigma_s,ss reference: {len(_ts)} frames')
else:
    print('NOTE: ss reference dumps missing -', F_PAIRS_REF.name, '/', F_TRAJREF.name)

if F_PAIRS.exists() and F_TRAJSTRESS.exists():
    ss_prod_ts, ss_prod_stack = compute_ss_stress(F_PAIRS, F_TRAJSTRESS, binWidth, z_coords)
    ss_prod_ts = np.array(ss_prod_ts)
    print(f'sigma_s,ss production: {len(ss_prod_ts)} frames')
else:
    print('NOTE: ss production dumps missing')

pp_ref_m = pp_ref_lo = pp_ref_hi = None
pp_prod_ts = None; pp_prod_stack = None
if F_PPAIRS_REF.exists() and F_TRAJREF.exists():
    _ts, _stk = compute_pp_stress(F_PPAIRS_REF, F_BONDS_REF, F_TRAJREF, binWidth, z_coords, BOND_SIGN)
    if len(_stk): pp_ref_m, pp_ref_lo, pp_ref_hi = mean_ci(_stk, ci_level)
    print(f'sigma_p,pp reference: {len(_ts)} frames')
if F_PPAIRS.exists() and F_TRAJSTRESS.exists():
    pp_prod_ts, pp_prod_stack = compute_pp_stress(F_PPAIRS, F_BONDS, F_TRAJSTRESS, binWidth, z_coords, BOND_SIGN)
    pp_prod_ts = np.array(pp_prod_ts)
    print(f'sigma_p,pp production: {len(pp_prod_ts)} frames')

In [ ]:
# ---- sigma_p,pp sign validation (group sigma_p_ref should track pp+kin) ----
# The group-based reference polymer stress (incl. bonds + 1/2 ps cross) should
# have the SAME sign and comparable magnitude as the pp reconstruction.  If the
# pp curve is mirrored about zero, set BOND_SIGN = -1 in the Config cell.
if pp_ref_m is not None:
    _ig = in_gel
    corr = np.corrcoef(pp_ref_m[_ig], sig_p_ref_m[_ig])[0, 1]
    print(f'corr(sigma_p,pp_ref, group sigma_p_ref) inside gel = {corr:+.3f}')
    if corr < 0:
        print('  >>> NEGATIVE correlation: sigma_p,pp likely sign-flipped. '
              'Set BOND_SIGN = -1.0 in Config and re-run.')
    else:
        print('  sign looks consistent (positive correlation).')
else:
    print('pp reference unavailable - skipping sign check.')

## Plotting helpers

In [ ]:
def _fmt_val_unc(v, u):
    """value ± uncertainty, uncertainty rounded to 2 sig figs (value matched)."""
    v = float(v)
    if np.isfinite(u) and u > 0:
        dec = int(np.clip(1 - np.floor(np.log10(u)), 0, 6))
        return f'{v:.{dec}f} \u00b1 {u:.{dec}f}'
    return f'{v:.3g}'

def _fmt_mu(vals):
    """mean ± std (2 sig figs) over finite values."""
    a = np.asarray(vals, float); a = a[np.isfinite(a)]
    if a.size == 0: return 'n/a'
    return _fmt_val_unc(np.mean(a), np.std(a))

def _means_text(z, curve, in_gel):
    """Mean ± scatter of `curve` inside and outside the gel as formatted text."""
    mi = np.nanmean(curve[in_gel]) if in_gel.any() else np.nan
    mo = np.nanmean(curve[~in_gel]) if (~in_gel).any() else np.nan
    return mi, mo, f'mean in gel = {_fmt_mu(curve[in_gel])}\nmean out gel = {_fmt_mu(curve[~in_gel])}'

def _final_flat_inside(curve, z, in_gel, tol):
    """Flat = small linear TREND across the gel (noise-robust, unlike raw std).
    Returns True when |slope|*interior_width / |mean| < tol."""
    v = curve[in_gel]; zz = np.asarray(z)[in_gel]
    m = np.isfinite(v); v, zz = v[m], zz[m]
    if len(v) < 3: return False
    slope = np.polyfit(zz, v, 1)[0]
    rise = abs(slope) * (zz.max() - zz.min())
    denom = max(abs(np.mean(v)), 1e-9)
    return (rise / denom) < tol

def shade_gel(ax):
    ax.axvspan(zn(z_gel_lo), zn(z_gel_hi), **GEL_SHADE)

def mark_walls(ax, ts=None):
    """Frozen support (type4, solid) + piston (type5, dash-dot).  Lines only,
    no text labels.  ts=None -> reference (uncompressed) piston position;
    ts given -> piston per evolution timestep at its COMPRESSED position, with
    the final one dark and earlier ones faded."""
    if np.isfinite(z_support):
        ax.axvline(zn(z_support), color=WONG['black'], ls='-', lw=1.5, alpha=0.85, zorder=4)
    if ts is None:
        pistons = [(z_piston, True)]
    else:
        ts = np.asarray(ts)
        pistons = [(piston_z_at(t), i == len(ts) - 1) for i, t in enumerate(ts)]
    for pz, is_last in pistons:
        if np.isfinite(pz):
            ax.axvline(zn(pz), color=('0.15' if is_last else '0.7'), ls='-.',
                       lw=(1.6 if is_last else 1.0), alpha=(0.9 if is_last else 0.35),
                       zorder=(4 if is_last else 3))

def plot_reference(ax, z, m, lo, hi, color, ylabel, title, annotate=True):
    zx = zn(z)
    ax.fill_between(zx, lo, hi, color=color, alpha=0.25, lw=0, zorder=2)
    ax.plot(zx, m, '-', color=color, lw=2.5, zorder=3)
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    shade_gel(ax); mark_walls(ax)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0, 1); ax.grid(alpha=0.3)
    if annotate:
        mi, mo, txt = _means_text(z, m, in_gel)
        ax.text(0.02, 0.03, txt, transform=ax.transAxes, va='bottom', ha='left',
                fontsize=15, bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

def subsample(ts, stack, k):
    """Evenly pick up to k snapshots (keep order; always include the last)."""
    n = len(ts)
    if n <= k: idx = np.arange(n)
    else:      idx = np.unique(np.linspace(0, n-1, k).round().astype(int))
    return ts[idx], stack[idx]

def plot_evolution(ax, z, ts, stack, ylabel, title):
    """Time-coloured profiles (cividis); final curve bold black; gel shaded.
    Mean-in/out annotation ONLY if the final curve is flat inside the gel."""
    ts = np.asarray(ts); stack = np.asarray(stack)
    norm = Normalize(vmin=ts.min(), vmax=ts.max())
    cmap = plt.get_cmap(EVO_CMAP)
    zx = zn(z)
    for i in range(len(ts)):
        is_last = (i == len(ts)-1)
        ax.plot(zx, stack[i], '-',
                color=('k' if is_last else cmap(norm(ts[i]))),
                lw=(3.5 if is_last else 1.6),
                alpha=(1.0 if is_last else 0.75),
                zorder=(5 if is_last else 3))
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    shade_gel(ax); mark_walls(ax, ts)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0, 1); ax.grid(alpha=0.3)
    sm = plt.cm.ScalarMappable(cmap=EVO_CMAP, norm=norm); sm.set_array([])
    cb = ax.figure.colorbar(sm, ax=ax, fraction=0.046, pad=0.02); cb.set_label('timestep')
    # gel is squeezed between the support and the CURRENT piston: restrict the
    # interior to that window so the reservoir past the piston never enters the
    # flatness test or the mean (that jump is what made noisy panels read "not flat").
    z_pist = piston_z_at(np.asarray(ts)[-1])
    _zz = np.asarray(z)
    interior = in_gel & (_zz >= z_gel_lo + wall_margin) & (_zz <= z_pist - wall_margin)
    if _final_flat_inside(stack[-1], z, interior, flat_tol):
        ax.text(0.02, 0.03, 'final (equilibrated)\nmean in gel = ' + _fmt_mu(stack[-1][interior]),
                transform=ax.transAxes, va='bottom', ha='left', fontsize=14,
                bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))
    else:
        ax.text(0.02, 0.03, 'final not flat inside\n(means omitted)', transform=ax.transAxes,
                va='bottom', ha='left', fontsize=13, color='0.35')

def robust_ylim(ax, curves, zmask=None, pad=0.12, qlo=2, qhi=98, include_zero=True):
    """Frame the y-axis to the bulk of the data (qlo–qhi percentiles), ignoring
    a few extreme wall-edge spike bins.  curves: iterable of 1-D arrays (each a
    profile); zmask: optional boolean mask selecting which bins to consider."""
    vals = []
    for c in curves:
        c = np.asarray(c, float)
        if zmask is not None: c = c[zmask]
        c = c[np.isfinite(c)]
        if c.size: vals.append(c)
    if not vals:
        return
    v = np.concatenate(vals)
    lo, hi = np.percentile(v, [qlo, qhi])
    if include_zero:
        lo, hi = min(lo, 0.0), max(hi, 0.0)
    if hi <= lo: hi = lo + 1.0
    d = (hi - lo) * pad
    ax.set_ylim(lo - d, hi + d)

print('Plot helpers ready')

## 0 — Combined sweep overview (all strain levels, shown first)

These panels overlay **every** `COMP_LEVELS` level on shared axes so the whole
stress–strain sweep is visible at a glance, before the detailed single-level
sections below.  They are produced **only when `COMP_LEVELS` has more than one
level** (`IS_SWEEP`); for a single run this section is skipped and you drop
straight into the per-level plots.

Colour = strain level (see legend).  Within each level the relaxation evolution
is drawn faint→bold (early holds transparent, the final equilibrated profile
thick and opaque).  Requires the per-level files to be synced first (run the
sync cell above — it now pulls every level).

In [ ]:
# ==========================================================================
#  0 — COMBINED SWEEP: load every level once, into SWEEP (list of dicts)
# ==========================================================================
# Self-contained: reuses the readers/helpers defined above but builds its own
# per-level paths and its own z-grid (the box is fixed, only the piston moves,
# so every level shares Z_LO/LZ).  ss stress and piston data are optional per
# level and degrade gracefully if a file is missing.
from matplotlib.lines import Line2D

IS_SWEEP = len(COMP_LEVELS) > 1

# categorical, CVD-safe colour per level (Wong palette, cycled if >8 levels)
LEVEL_COLORS = [WONG[k] for k in ('blue','vermillion','green','reddishpurple',
                                  'orange','skyblue','yellow','black')]
def level_color(i): return LEVEL_COLORS[i % len(LEVEL_COLORS)]

def _box_area_from_header(dumpfile):
    with open(dumpfile) as fh:
        head = [next(fh) for _ in range(9)]
    xlo, xhi = map(float, head[5].split()); ylo, yhi = map(float, head[6].split())
    return (xhi - xlo) * (yhi - ylo)

def _net_from_total(sig_p_list, sig_s_list, zc):
    """Terzaghi split per level (mirrors section 3): network = total - pore
    baseline (flat reservoir at z/Lz~0.95); membrane from final polymer stress."""
    tot = np.asarray([sig_p_list[i] + sig_s_list[i] for i in range(len(sig_p_list))])
    zf = (zc - zc.min()) / (zc.max() - zc.min())
    bw = (zf >= 0.91) & (zf <= 0.99) & (zf < 0.995)
    spf = np.abs(sig_p_list[-1]); thr = gel_thresh * float(np.nanmax(spf))
    mem = spf > thr
    zlo = float(zc[mem].min()) if mem.any() else zc[0]
    zhi = float(zc[mem].max()) if mem.any() else zc[-1]
    in_mem = (zc >= zlo) & (zc <= zhi)
    net = np.zeros_like(tot); pore = np.zeros(len(tot)); pore_h = np.zeros(len(tot))
    for i in range(len(tot)):
        v = tot[i][bw]; v = v[np.isfinite(v)]
        p0 = float(np.nanmean(v)) if len(v) else 0.0
        se = float(stats.sem(v)) if len(v) > 1 else 0.0
        tcr = stats.t.ppf(0.5 + ci_level/2, df=max(len(v)-1, 1))
        pore[i] = p0; pore_h[i] = tcr * se; net[i] = tot[i] - p0
    return tot, net, pore, pore_h, in_mem, (zlo, zhi)

def load_level(lvl):
    b = f'{sim_name}_c{lvl}'
    P  = lambda n, e='dat': DATA_DIR / f'{n}_{b}.{e}'
    Tf = lambda n: TRAJ_DIR / f'{n}_{b}.lammpstrj'
    fp, fs = P('sigmazz_polymer'), P('sigmazz_solvent')
    if not (fp.exists() and fs.exists()):
        print(f'  level {lvl}: sigmazz files missing — skipping'); return None
    sp, ss = read_ave_time_file(fp), read_ave_time_file(fs)
    if not sp or not ss:
        print(f'  level {lvl}: empty sigmazz files — skipping'); return None
    ts = np.array([x[0] for x in sp])
    sig_p = [x[2] for x in sp]; sig_s = [x[2] for x in ss]
    zc = Z_LO + sp[0][1] * binWidth - binWidth/2.0
    tot, net, pore, pore_h, in_mem, memb = _net_from_total(sig_p, sig_s, zc)
    L = dict(lvl=lvl, ts=ts, z=zc, tot=tot, net=net, pore=pore, pore_h=pore_h,
             in_mem=in_mem, memb=memb)
    # solvent density + mass fraction
    fd = P('solvent_density_z')
    if fd.exists():
        d_ts, d_z, d_n, d_m = _load_density(fd)
        # Normalise mass fraction by THIS level's OWN bulk reservoir density (the
        # coexisting bath, z/Lz in [0.90, 0.99] of the final snapshot) so every
        # reservoir plateau lands at 1.0.  A single global rho_s0 leaves each
        # level slightly off 1 because the reservoir density itself drifts as the
        # gel consolidates and the reservoir grows.
        _zf = (d_z - d_z.min()) / (d_z.max() - d_z.min())
        _res = (_zf >= 0.90) & (_zf <= 0.99)
        rho_res = float(np.nanmean(d_m[-1][_res])) if _res.any() else rho_s0
        L.update(d_ts=d_ts, d_z=d_z, d_m=d_m, d_mf=d_m / rho_res, rho_res=rho_res)
    # solvent-only ss stress (needs pair dump + stress traj)
    fpairs, ftr = P('pairs','dump'), Tf('traj_stress')
    if fpairs.exists() and ftr.exists():
        try:
            _t, _stk = compute_ss_stress(fpairs, ftr, binWidth, zc)
            if len(_stk): L['ss_ts'], L['ss'] = np.array(_t), _stk
        except Exception as e:
            print(f'  level {lvl}: ss stress failed ({e})')
    # strain
    fstr = P('strain_zz')
    if fstr.exists():
        s_ts, s_eps = read_strain_file(fstr)
        L['eps'] = float(s_eps[-1]); L['eps_ts'] = s_ts; L['eps_series'] = s_eps
    # cross-section area (lx*ly): from box_dimensions tail, else the ref box header
    fb = P('box_dimensions'); area = None
    if fb.exists():
        B = np.loadtxt(fb, comments='#')
        if B.ndim == 2 and B.shape[1] >= 3: area = float(np.mean(B[-5:,1]*B[-5:,2]))
    if area is None and F_PAIRS_REF.exists():
        area = _box_area_from_header(F_PAIRS_REF)
    L['area'] = area
    # piston force / pressure
    ff = P('piston_force')
    if ff.exists():
        pf = read_print_file(ff, col_names=['step','Fz'])
        L['pf_step'], L['pf_F'] = pf['step'].astype(int), pf['Fz']
        if area: L['pf_P'] = pf['Fz'] / area
    ffa = P('piston_force_avg')
    if ffa.exists() and area:
        pfa = read_print_file(ffa, col_names=['step','Fz'])
        L['pfa_step'], L['pfa_P'] = pfa['step'].astype(int), pfa['Fz'] / area
    # longitudinal modulus, two estimates
    if L.get('eps', 0) > 0:
        mn = net[-1][in_mem] / L['eps']; mn = mn[np.isfinite(mn)]
        if len(mn):
            L['M_net'], L['M_net_lo'], L['M_net_hi'] = mean_ci(mn, ci_level)
        if area and 'pf_F' in L:
            Pf = float(np.mean(L['pfa_P'][-5:])) if 'pfa_P' in L and len(L['pfa_P']) \
                 else float(np.mean(L['pf_P'][-5:]))
            L['M_pist'] = Pf / L['eps']
            Fw = L['pf_F'][-10:]
            L['M_pist_err'] = float(np.std(Fw)/(area*L['eps'])) if len(Fw) > 1 else 0.0
            L['P_final'] = Pf
    return L

if IS_SWEEP:
    print(f'Loading {len(COMP_LEVELS)} sweep levels: {COMP_LEVELS}')
    SWEEP = [L for L in (load_level(l) for l in COMP_LEVELS) if L is not None]
    SWEEP.sort(key=lambda L: float(L['lvl']))
    print(f'>>> loaded {len(SWEEP)} level(s): {[L["lvl"] for L in SWEEP]}')
else:
    SWEEP = []
    print('Single run (one level in COMP_LEVELS) — combined sweep plots below are skipped.')

In [ ]:
# ==========================================================================
#  0 — COMBINED SWEEP: profile-evolution overlays (all levels on shared axes)
# ==========================================================================
def overlay_evolution(ax, zkey, tskey, stackkey, ylabel, title,
                      autoscale_mask=None):
    """Overlay each level's relaxation evolution.  Colour = level; within a
    level alpha ramps early->late and the final curve is bold."""
    all_curves = []
    for i, L in enumerate(SWEEP):
        if stackkey not in L or L[stackkey] is None or not len(L[stackkey]):
            continue
        ts, stack = subsample(np.asarray(L[tskey]), np.asarray(L[stackkey]), n_curves)
        zx = zn(L[zkey]); col = level_color(i); nc = len(ts)
        for j in range(nc):
            last = (j == nc - 1)
            ramp = 0.20 + 0.80 * (j / max(nc - 1, 1))
            ax.plot(zx, stack[j], '-', color=col,
                    lw=(3.0 if last else 1.1),
                    alpha=(1.0 if last else 0.55 * ramp),
                    zorder=(5 if last else 3))
            if last: all_curves.append(stack[j])
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
    ax.set_xlim(0, 1); ax.grid(alpha=0.3)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(ylabel); ax.set_title(title)
    if all_curves:
        robust_ylim(ax, all_curves, zmask=autoscale_mask, pad=0.15)

if IS_SWEEP and SWEEP:
    handles = [Line2D([0],[0], color=level_color(i), lw=3,
                      label=fr'$\varepsilon_\mathrm{{target}}={float(L["lvl"]):.2f}$')
               for i, L in enumerate(SWEEP)]

    fig, ax = plt.subplots(2, 3, figsize=(22, 12), constrained_layout=True)
    fig.suptitle(f'Combined sweep — profile evolution, all levels:  {sim_name}',
                 fontsize=15, fontweight='bold')

    # membrane-interior + reservoir mask (box-fixed) for network-stress autoscale
    _z0 = SWEEP[0]['z']
    _mem_lo = min(L['memb'][0] for L in SWEEP)
    _net_mask = (_z0 >= _mem_lo + wall_margin) & (_z0 <= _z0.max())

    overlay_evolution(ax[0,0], 'z','ts','tot',
                      r'$\sigma_{zz}^{t}(z,t)$', r'(a) Total stress $\sigma_{zz}^{t}$')
    overlay_evolution(ax[0,1], 'z','ss_ts','ss',
                      r'$\sigma_{s,zz}^{ss}(z,t)$', r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$')
    overlay_evolution(ax[0,2], 'd_z','d_ts','d_m',
                      r'$\rho_s(z,t)\ (m\,\sigma^{-3})$', r'(c) Solvent mass density $\rho_s$')
    overlay_evolution(ax[1,0], 'd_z','d_ts','d_mf',
                      r'$\rho_s/\rho_{s,0}$', r'(d) Solvent mass fraction $\rho_s/\rho_{s,0}$')
    ax[1,0].axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)
    overlay_evolution(ax[1,1], 'z','ts','net',
                      r"$\sigma'_{zz}(z,t)$", r"(e) Network stress $\sigma'_{zz}$",
                      autoscale_mask=_net_mask)

    # (f) pore pressure: flat baseline per level -> final value vs strain target
    axf = ax[1,2]
    for i, L in enumerate(SWEEP):
        eps = L.get('eps', float(L['lvl']))
        axf.errorbar([eps], [L['pore'][-1]], yerr=[L['pore_h'][-1]], fmt='o', ms=11,
                     color=level_color(i), capsize=6, lw=2)
        # label with the nominal target; the measured strain overshoots it because
        # the network keeps consolidating during the frozen-piston hold.
        axf.annotate(fr'target {L["lvl"]}', (eps, L['pore'][-1]),
                     textcoords='offset points', xytext=(7, 4), fontsize=10,
                     color=level_color(i))
    axf.set_xlabel(r'gel strain  $\varepsilon$'); axf.set_ylabel(r'$p_{\mathrm{pore}}$ (final)')
    axf.set_title(r'(f) Pore pressure (equilibrated) vs strain'); axf.grid(alpha=0.3)

    for a in (ax[0,0], ax[0,1], ax[0,2], ax[1,0], ax[1,1]):
        a.legend(handles=handles, fontsize=11, loc='best', title='sweep level')

    out = PLOT_DIR / f'sweep_profiles_combined_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('combined profile overlays skipped (not a sweep / no levels loaded)')

In [ ]:
# ==========================================================================
#  0 — COMBINED SWEEP: piston pressure & force histories (all levels)
# ==========================================================================
def _roll(y, win=21):
    y = np.asarray(y, float); n = len(y)
    if win <= 1 or n == 0: return y.copy()
    half = win // 2; csum = np.concatenate(([0.0], np.cumsum(y))); out = np.empty(n)
    for k in range(n):
        lo, hi = max(0, k-half), min(n, k+half+1); out[k] = (csum[hi]-csum[lo])/(hi-lo)
    return out

if IS_SWEEP and SWEEP:
    fig, (axP, axF) = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
    fig.suptitle(f'Combined sweep — piston pressure & force histories:  {sim_name}',
                 fontsize=15, fontweight='bold')
    handles = []
    for i, L in enumerate(SWEEP):
        col = level_color(i)
        lab = fr'$\varepsilon_\mathrm{{target}}={float(L["lvl"]):.2f}$'
        if 'pf_P' in L:
            axP.plot(L['pf_step'], L['pf_P'], '-', color=col, lw=0.8, alpha=0.20)
            axP.plot(L['pf_step'], _roll(L['pf_P']), '-', color=col, lw=2.4, alpha=0.95)
        if 'pf_F' in L:
            axF.plot(L['pf_step'], L['pf_F'], '-', color=col, lw=0.8, alpha=0.20)
            axF.plot(L['pf_step'], _roll(L['pf_F']), '-', color=col, lw=2.4, alpha=0.95)
        handles.append(Line2D([0],[0], color=col, lw=3, label=lab))
    axP.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    axP.set_xlabel('step'); axP.set_ylabel(r'$P = F_z/A$  (LJ / $\sigma^2$)')
    axP.set_title('(a) piston pressure vs step'); axP.grid(alpha=0.3)
    axF.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    axF.set_xlabel('step'); axF.set_ylabel(r'$F_z$  (LJ)')
    axF.set_title('(b) piston force vs step'); axF.grid(alpha=0.3)
    axP.legend(handles=handles, fontsize=11, loc='best', title='sweep level')
    out = PLOT_DIR / f'sweep_piston_history_combined_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('combined piston histories skipped (not a sweep / no levels loaded)')

In [ ]:
# ==========================================================================
#  0 — COMBINED SWEEP: longitudinal modulus M (both estimates) vs strain
# ==========================================================================
# M_network = <sigma'_zz>_membrane / eps   and   M_piston = (F_z/A) / eps,
# one point per level.  Agreement across strains validates the Terzaghi split;
# the initial secant slope of P vs eps is the small-deformation modulus.
if IS_SWEEP and SWEEP:
    eps = np.array([L.get('eps', float(L['lvl'])) for L in SWEEP])
    have_net  = [('M_net'  in L) for L in SWEEP]
    have_pist = [('M_pist' in L) for L in SWEEP]

    fig, (axM, axPP) = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    fig.suptitle(f'Combined sweep — longitudinal modulus & stress–strain:  {sim_name}',
                 fontsize=15, fontweight='bold')

    # (a) the two M estimates vs strain
    en = eps[have_net]
    if any(have_net):
        Mn  = np.array([L['M_net']    for L in SWEEP if 'M_net' in L])
        Mnl = np.array([L['M_net_lo'] for L in SWEEP if 'M_net' in L])
        Mnh = np.array([L['M_net_hi'] for L in SWEEP if 'M_net' in L])
        axM.errorbar(en, Mn, yerr=[Mn-Mnl, Mnh-Mn], fmt='o-', ms=10, lw=2,
                     color=WONG['blue'], capsize=6,
                     label=r"network  $\langle\sigma'_{zz}\rangle_\mathrm{mem}/\varepsilon$")
    if any(have_pist):
        ep = eps[have_pist]
        Mp  = np.array([L['M_pist']     for L in SWEEP if 'M_pist' in L])
        Mpe = np.array([L['M_pist_err'] for L in SWEEP if 'M_pist' in L])
        axM.errorbar(ep, Mp, yerr=Mpe, fmt='s-', ms=10, lw=2,
                     color=WONG['vermillion'], capsize=6,
                     label=r'piston  $(F_z/A)/\varepsilon$')
    axM.set_xlabel(r'gel strain  $\varepsilon$'); axM.set_ylabel(r'$M$  (LJ units)')
    axM.set_title('(a) longitudinal modulus, two estimates'); axM.grid(alpha=0.3)
    axM.legend(fontsize=13, loc='best')

    # (b) stress-strain: piston pressure vs strain (secant modulus = slope)
    if any(have_pist):
        ep = eps[have_pist]
        Pp = np.array([L['P_final'] for L in SWEEP if 'M_pist' in L])
        order = np.argsort(ep); ep, Pp = ep[order], Pp[order]
        axPP.plot(ep, Pp, 'o-', lw=2, color=WONG['green'])
        for i,(e,p) in enumerate(zip(ep,Pp)):
            axPP.annotate(f'{SWEEP[i]["lvl"]}', (e,p), textcoords='offset points',
                          xytext=(6,6), fontsize=12)
        if len(ep) >= 2:
            M_init = (Pp[1]-Pp[0])/(ep[1]-ep[0]); M_sec = (Pp[-1]-Pp[0])/(ep[-1]-ep[0])
            axPP.set_title(fr'(b) $P$ vs $\varepsilon$   $M_\mathrm{{init}}\approx{M_init:.3g}$,  '
                           fr'$M_\mathrm{{secant}}\approx{M_sec:.3g}$')
        else:
            axPP.set_title(r'(b) piston pressure vs strain')
    axPP.set_xlabel(r'gel strain  $\varepsilon$')
    axPP.set_ylabel(r'piston pressure  $P=\langle F_z\rangle/A$  (LJ)')
    axPP.grid(alpha=0.3)

    out = PLOT_DIR / f'sweep_modulus_combined_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('combined modulus plot skipped (not a sweep / no levels loaded)')

## 1 — Reference-state profiles (ε = 0) with confidence intervals

Averaged over the 150k pre-compression window (piston held at v = 0).  Bands are
the 95 % CIs across the reference snapshots; grey shading marks the gel interior.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle(f'Reference state (ε = 0):  {sim_name}', fontsize=14, fontweight='bold')

# (a) total sigma_zz = sigma_p,zz + sigma_s,zz
plot_reference(axes[0,0], z_coords, sig_t_ref_m, sig_t_ref_lo, sig_t_ref_hi,
               WONG['blue'], r'$\sigma_{zz}^{t}(z)$', r'(a) Total stress $\sigma_{zz}^{t}=\sigma_{p,zz}+\sigma_{s,zz}$')

# (b) solvent-only sigma_s,ss
if ss_ref_m is not None:
    plot_reference(axes[0,1], z_coords, ss_ref_m, ss_ref_lo, ss_ref_hi,
                   WONG['vermillion'], r'$\sigma_{s,zz}^{ss}(z)$', r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$')
else:
    axes[0,1].text(0.5,0.5,'ss reference\nunavailable',ha='center',va='center',transform=axes[0,1].transAxes)
    axes[0,1].set_title(r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$')

# (c) solvent mass density rho_s(z)
plot_reference(axes[1,0], dens_z, rho_ref_m, rho_ref_lo, rho_ref_hi,
               WONG['green'], r'$\rho_s(z)\ (m\,\sigma^{-3})$', r'(c) Solvent mass density $\rho_s$')
axes[1,0].axhline(rho_s0, color=WONG['black'], ls=':', lw=1.5, alpha=0.7)
axes[1,0].text(0.98, 0.05, r'$\rho_{s,0}$'+f' = {rho_s0:.3f}', transform=axes[1,0].transAxes,
               ha='right', va='bottom', fontsize=15)

# (d) mass fraction rho_s/rho_s0
plot_reference(axes[1,1], dens_z, mf_ref_m, mf_ref_lo, mf_ref_hi,
               WONG['reddishpurple'], r'$\rho_s/\rho_{s,0}$', r'(d) Solvent mass fraction $\rho_s/\rho_{s,0}$')
axes[1,1].axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)

out = PLOT_DIR / f'reference_state_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

## 2 — Profile evolution vs. timestep (relaxation phase)

Each observable sampled as up to 10 curves **starting after** the 10 % compression
halt (step shown above).  Colour = timestep (cividis); the **bold black** curve is
the final (most-relaxed) profile.  Gel interior shaded.  Per the request, no extra
time-averaged curve is drawn, and in/out-gel means are written only when the final
curve is flat inside the gel (i.e. has equilibrated).

In [ ]:
# Build post-halt evolution series for each observable, subsampled to ~n_curves.
# Total sigma_zz and density use their own snapshot grids; ss/pp use pair-dump grid.
def post_halt(ts, stack):
    ts = np.asarray(ts); stack = np.asarray(stack)
    m = ts >= halt_ts
    if m.sum() < 2:            # halt at/after last snapshot -> use last few frames
        m = np.zeros(len(ts), bool); m[-min(len(ts), n_curves):] = True
    return subsample(ts[m], stack[m], n_curves)

# total sigma_zz
tzz_ts, tzz_ev = post_halt(prod_ts, np.array(sig_t_zz))
# solvent mass density + mass fraction
dlm = dens_ts >= 0   # all
dmf_stack = dens_m / rho_s0
rho_ts, rho_ev = post_halt(dens_ts, dens_m)
mf_ts,  mf_ev  = post_halt(dens_ts, dmf_stack)

fig, axes = plt.subplots(2, 2, figsize=(17, 12), constrained_layout=True)
fig.suptitle(f'Profile evolution after {halt_ts}-step compression halt:  {sim_name}',
             fontsize=14, fontweight='bold')

plot_evolution(axes[0,0], z_coords, tzz_ts, tzz_ev,
               r'$\sigma_{zz}^{t}(z,t)$', r'(a) Total stress $\sigma_{zz}^{t}$')

if ss_prod_stack is not None and len(ss_prod_stack):
    ss_ts2, ss_ev = post_halt(ss_prod_ts, ss_prod_stack)
    plot_evolution(axes[0,1], z_coords, ss_ts2, ss_ev,
                   r'$\sigma_{s,zz}^{ss}(z,t)$', r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$')
else:
    axes[0,1].text(0.5,0.5,'ss production\nunavailable',ha='center',va='center',transform=axes[0,1].transAxes)

plot_evolution(axes[1,0], dens_z, rho_ts, rho_ev,
               r'$\rho_s(z,t)\ (m\,\sigma^{-3})$', r'(c) Solvent mass density $\rho_s$')

plot_evolution(axes[1,1], dens_z, mf_ts, mf_ev,
               r'$\rho_s/\rho_{s,0}$', r'(d) Solvent mass fraction $\rho_s/\rho_{s,0}$')
axes[1,1].axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)

out = PLOT_DIR / f'profile_evolution_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

### Optional — polymer-only stress σ_p,pp evolution

Provided separately since it is the new observable.  Same conventions as above.
If σ_p,pp looks sign-flipped relative to the group polymer stress (see the
validation cell), set `BOND_SIGN = -1.0` in Config and re-run.

In [ ]:
if pp_prod_stack is not None and len(pp_prod_stack):
    fig, axes = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    if pp_ref_m is not None:
        plot_reference(axes[0], z_coords, pp_ref_m, pp_ref_lo, pp_ref_hi,
                       WONG['orange'], r'$\sigma_{p,zz}^{pp}(z)$',
                       r'Reference $\sigma_{p,zz}^{pp}$ (pair + bond, virial only)')
    else:
        axes[0].text(0.5,0.5,'pp reference unavailable',ha='center',va='center',transform=axes[0].transAxes)
    pp_ts2, pp_ev = post_halt(pp_prod_ts, pp_prod_stack)
    plot_evolution(axes[1], z_coords, pp_ts2, pp_ev,
                   r'$\sigma_{p,zz}^{pp}(z,t)$', r'Evolution $\sigma_{p,zz}^{pp}$')
    out = PLOT_DIR / f'polymer_only_stress_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('sigma_p,pp production data unavailable - skipping.')

## 3 — Network (effective) stress & pore pressure

Terzaghi decomposition $\sigma^{t}_{zz}=\sigma'_{zz}+p_{\rm pore}$.  The **network stress** $\sigma'_{zz}$ is the jump of the total stress above the pore-fluid baseline (the barostat pressure, read per-curve from the flat reservoir at $z/L_z\approx0.95$); the **pore pressure** is total $-$ network.  Colour = timestep, **bold black** = final (equilibrated) curve; the first (fast-piston) curve is handled with fixed membrane bounds and a fixed baseline window. 95% error bands on every curve.

In [ ]:
# ==========================================================================
#  3 — NETWORK (EFFECTIVE) STRESS  σ'_zz  &  PORE PRESSURE  (evolution)
# ==========================================================================
# Terzaghi decomposition of the total normal stress inside the membrane:
#       σ_zz^t(z,t) = σ'_zz(z,t) + p_pore(z,t)
# The network (effective) stress is the JUMP of the total stress above the
# pore-fluid baseline; the pore pressure is then total − network.
#
# Baseline (pore) pressure is read PER CURVE from the flat far-reservoir around
# z/Lz = BASELINE_ZF — this is the barostat/bath pressure (≈ P_target = 1.5 for
# a P=1.5 run).  z/Lz≈0.95 stays flat even for the fast-piston FIRST curve
# (the piston slope sits between the membrane and this window), so the
# subtraction is clean for every timestep.
#
# FIXED bounds (per the request): the membrane extent is determined ONCE from
# the final equilibrated polymer stress and reused for ALL timesteps, so the
# fast piston in the first curve cannot distort a per-curve gel detection.
BASELINE_ZF      = 0.95     # fractional z (z/Lz) of the reservoir baseline window
BASELINE_ZF_HALF = 0.04     # half-width of that window (fraction of Lz)

zf = (z_coords - z_coords.min()) / (z_coords.max() - z_coords.min())
_bw = (zf >= BASELINE_ZF - BASELINE_ZF_HALF) & (zf <= BASELINE_ZF + BASELINE_ZF_HALF)
_bw &= (zf < 0.995)          # drop the half-empty extreme-edge bin (stress rolls off)
print(f'pore baseline window: z/Lz in [{BASELINE_ZF-BASELINE_ZF_HALF:.2f}, '
      f'{BASELINE_ZF+BASELINE_ZF_HALF:.2f}]  ({_bw.sum()} bins, '
      f'z in [{z_coords[_bw].min():.0f}, {z_coords[_bw].max():.0f}])')

# ---- FIXED membrane bounds from the final (equilibrated) polymer stress ----
# Piston is out of the membrane by the final curve, so this is the clean extent.
_spf = np.abs(sig_p_zz[-1]); _thr = gel_thresh * float(np.nanmax(_spf))
_mem = _spf > _thr
z_mem_lo = float(z_coords[_mem].min()) if _mem.any() else z_gel_lo
z_mem_hi = float(z_coords[_mem].max()) if _mem.any() else z_gel_hi
in_mem = (z_coords >= z_mem_lo) & (z_coords <= z_mem_hi)
print(f'membrane (FIXED for all curves, from final polymer stress): '
      f'z in [{z_mem_lo:.1f}, {z_mem_hi:.1f}]  ({in_mem.sum()} bins)')

# ---- per-bin single-snapshot noise, estimated from the reference window ----
# A single fix ave/time vector carries no per-bin variance; the reference
# snapshots share the same nfreq averaging, so their per-bin scatter is a fair
# proxy for the statistical noise of ONE production σ_zz^t snapshot.
sd_bin = np.nanstd(ref_t_stack, axis=0)
# Fallback when the reference window is too thin (<2 snapshots) to estimate
# per-bin scatter: use the spatial scatter of the total stress in the flat
# reservoir baseline window as a uniform noise floor so error bars never vanish.
if ref_t_stack.shape[0] < 2 or not np.any(sd_bin > 0):
    _floor = float(np.nanstd(np.asarray(sig_t_zz)[-1][_bw]))
    sd_bin = np.full(len(z_coords), _floor)
    print(f'  (thin reference: using reservoir noise floor sd_bin = {_floor:.4f})')
_z95   = 1.959963985                                  # 95% normal quantile

# ---- network stress + pore pressure for every production snapshot ----
tot_stack = np.asarray(sig_t_zz)                      # (n_prod, nz)
net_stack = np.zeros_like(tot_stack)
pore_val  = np.zeros(len(tot_stack))                  # scalar baseline per curve
pore_half = np.zeros(len(tot_stack))                  # 95% half-width of the baseline
for i in range(len(tot_stack)):
    v  = tot_stack[i][_bw]; v = v[np.isfinite(v)]
    p0 = float(np.nanmean(v))
    se = float(stats.sem(v)) if len(v) > 1 else 0.0
    tcr = stats.t.ppf(0.5 + ci_level/2, df=max(len(v)-1, 1))
    pore_val[i]  = p0
    pore_half[i] = tcr * se
    net_stack[i] = tot_stack[i] - p0                  # σ'_zz = σ^t − p_pore
# network 95% band: per-bin snapshot noise ⊕ baseline uncertainty (in quadrature)
net_half = np.sqrt((_z95 * sd_bin[None, :])**2 + (pore_half[:, None])**2)

print(f"pore pressure (baseline)  t={prod_ts[0]}→{prod_ts[-1]}: "
      f"{pore_val[0]:.3f} → {pore_val[-1]:.3f}")
print(f"network σ'_zz in membrane t={prod_ts[0]}→{prod_ts[-1]}: "
      f"{np.nanmean(net_stack[0][in_mem]):.3f} → {np.nanmean(net_stack[-1][in_mem]):.3f}")


In [ ]:
# ---- evolution plots: network (effective) stress + pore pressure ----
# Same conventions as the other evolution plots: colour = timestep (cividis),
# the bold black curve is the final (equilibrated) profile, membrane shaded.
# The FIRST curve is the fast-piston / actively-compressed state; because the
# membrane bounds and the z/Lz≈0.95 baseline are fixed, it is handled cleanly.
def _shade_mem(ax):
    ax.axvspan(zn(z_mem_lo), zn(z_mem_hi), **GEL_SHADE)

net_ts, net_ev  = subsample(prod_ts, net_stack, n_curves)
_,      neth_ev = subsample(prod_ts, net_half,  n_curves)
_,      porev   = subsample(prod_ts, pore_val,  n_curves)
_,      poreh   = subsample(prod_ts, pore_half, n_curves)

fig, axes = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
fig.suptitle(f'Network stress & pore pressure evolution:  {sim_name}',
             fontsize=14, fontweight='bold')
norm = Normalize(vmin=net_ts.min(), vmax=net_ts.max()); cmap = plt.get_cmap(EVO_CMAP)
zc = zn(z_coords)

# (a) network (effective) stress σ'_zz(z,t)
axN = axes[0]
for i in range(len(net_ts)):
    last = (i == len(net_ts) - 1)
    c = 'k' if last else cmap(norm(net_ts[i])); lw = 3.5 if last else 1.6
    axN.fill_between(zc, net_ev[i]-neth_ev[i], net_ev[i]+neth_ev[i],
                     color=c, alpha=(0.20 if last else 0.06), lw=0, zorder=(4 if last else 2))
    axN.plot(zc, net_ev[i], '-', color=c, lw=lw,
             alpha=(1.0 if last else 0.8), zorder=(5 if last else 3))
axN.axhline(0, color='k', ls='--', lw=1, alpha=0.5); _shade_mem(axN); mark_walls(axN, net_ts)
axN.set_xlabel(r'$z/L_z$'); axN.set_ylabel(r"$\sigma'_{zz}(z,t)$")
axN.set_title(r"(a) Network stress $\sigma'_{zz}=\sigma^{t}_{zz}-p_{\mathrm{pore}}$")
axN.set_xlim(0, 1); axN.grid(alpha=0.3)
# y-limits framed to the membrane interior + reservoir (drop the support-side
# wall spike; percentiles absorb the piston-interface spike) so the ~0.8 membrane
# value and the ~0 reservoir are both visible instead of being clipped.
_net_ymask = (z_coords >= z_mem_lo + wall_margin) & (z_coords <= z_coords.max())
robust_ylim(axN, list(net_ev), zmask=_net_ymask, pad=0.15)
axN.text(0.02, 0.03, 'final (equilibrated)\n'
         f'mean in membrane = {_fmt_mu(net_ev[-1][in_mem])}',
         transform=axN.transAxes, va='bottom', ha='left', fontsize=14,
         bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

# (b) pore pressure p_pore(z,t) = total − network = flat baseline
axP = axes[1]
_zends = [0.0, 1.0]
for i in range(len(net_ts)):
    last = (i == len(net_ts) - 1)
    c = 'k' if last else cmap(norm(net_ts[i])); lw = 3.5 if last else 1.6
    axP.fill_between(_zends, [porev[i]-poreh[i]]*2, [porev[i]+poreh[i]]*2,
                     color=c, alpha=(0.20 if last else 0.06), lw=0, zorder=(4 if last else 2))
    axP.plot(_zends, [porev[i]]*2, '-', color=c, lw=lw,
             alpha=(1.0 if last else 0.8), zorder=(5 if last else 3))
_shade_mem(axP); mark_walls(axP, net_ts)
axP.set_xlabel(r'$z/L_z$'); axP.set_ylabel(r'$p_{\mathrm{pore}}(z,t)$')
axP.set_title(r"(b) Pore pressure $p_{\mathrm{pore}}=\sigma^{t}_{zz}-\sigma'_{zz}$")
axP.set_xlim(0, 1); axP.grid(alpha=0.3)
axP.text(0.02, 0.03, 'final (equilibrated)\n'
         f'$p_{{\\mathrm{{pore}}}}$ = {_fmt_val_unc(porev[-1], poreh[-1])}',
         transform=axP.transAxes, va='bottom', ha='left', fontsize=14,
         bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

sm = plt.cm.ScalarMappable(cmap=EVO_CMAP, norm=norm); sm.set_array([])
cb = fig.colorbar(sm, ax=axes, fraction=0.046, pad=0.02); cb.set_label('timestep')

out = PLOT_DIR / f'network_stress_pore_pressure_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()


## 4 — Piston pressure history (relaxation check)

$P(t) = F_{z,\mathrm{piston}}(t)/A$ from the pairwise contact force on the piston
atoms recorded by `fix out_piston_force` in `triaxial_compression.lmp` (same
methodology as `slab_with_flow.lmp` compression mode).  The cross-section
$A = l_x l_y$ is fixed in Phase 2 (no lateral barostat), so it is read once from
the pair-dump box header.  The **log panel** makes it easy to see when the
network has relaxed: once $\ln P$ flattens, the piston pressure has reached its
equilibrium plateau.

In [ ]:
# ==========================================================================
#  4 — PISTON FORCE / PRESSURE  (from out_piston_force in triaxial_compression.lmp)
# ==========================================================================
# The .lmp records the summed pairwise z-force on the piston atoms:
#     fix out_piston_force all print ... '$(step) $(c_piston_fz)' ...
#         file .../piston_force_<sim_name>.dat
# Pressure P = F_z / A, A = lx*ly the FIXED box cross-section (no lateral
# barostat in Phase 2), read once from the pair-dump box header.
# F_PISTON_FORCE defined in the Config cell (also used by the Expanse sync cell).
def _read_box_xy(dumpfile):
    """Return (lx, ly) from the first frame of a LAMMPS dump header."""
    with open(dumpfile) as f:
        head = [next(f) for _ in range(9)]
    xlo, xhi = map(float, head[5].split())   # line 5 = x BOX BOUNDS
    ylo, yhi = map(float, head[6].split())   # line 6 = y BOX BOUNDS
    return (xhi - xlo), (yhi - ylo)

LX, LY = _read_box_xy(F_PAIRS_REF)
piston_area = LX * LY
print(f'piston area  A = lx*ly = {LX:.2f} x {LY:.2f} = {piston_area:.2f} sigma^2')

if Path(F_PISTON_FORCE).exists():
    _pf         = read_print_file(F_PISTON_FORCE, col_names=['step', 'F_piston_z'])
    steps_pf    = _pf['step'].astype(int)
    F_piston    = _pf['F_piston_z']
    P_piston_ts = F_piston / piston_area
    has_piston_force = True
    print(f'piston force: {len(steps_pf)} snapshots, steps '
          f'{steps_pf[0]} -> {steps_pf[-1]};  F_z in '
          f'[{F_piston.min():.2f}, {F_piston.max():.2f}] LJ;  '
          f'P in [{P_piston_ts.min():.4f}, {P_piston_ts.max():.4f}] LJ/sigma^2')
else:
    has_piston_force = False
    print(f'MISSING: {F_PISTON_FORCE.name} -- sync it from the cluster '
          '(output_files/piston_data/).  Piston-pressure plots will be skipped.')

# ---- LMP block-averaged piston force (optional; from fix out_piston_force_avg) ----
# Present only for runs made with the updated .lmp.  If absent, the plot below
# still smooths the raw series with a notebook-side rolling mean.
if Path(F_PISTON_FORCE_AVG).exists():
    _pfa            = read_print_file(F_PISTON_FORCE_AVG, col_names=['step', 'F_piston_z'])
    steps_pf_avg    = _pfa['step'].astype(int)
    F_piston_avg    = _pfa['F_piston_z']
    P_piston_avg_ts = F_piston_avg / piston_area
    has_piston_force_avg = True
    print(f'LMP block-averaged piston force: {len(steps_pf_avg)} snapshots '
          f'(pf_nevery x pf_nrepeat window)')
else:
    has_piston_force_avg = False
    print('LMP block-averaged piston force: not present '
          '(will smooth raw series with a rolling mean instead)')


In [ ]:
# ---- piston pressure P = F_z/A vs timestep: linear + log ----
# Raw c_piston_fz is thermally noisy (there is NO damping on the piston, by
# design).  We reduce the visual noise by TIME-AVERAGING, not damping:
#   - a notebook-side centered rolling mean of the raw series (always shown), and
#   - the LMP block-averaged series if the run has it (has_piston_force_avg).
roll_win = 21   # rolling-mean window in samples (odd; ~ roll_win*volume_freq steps)

def rolling_mean(y, win):
    """Centered moving average; window shrinks at the edges (min_periods=1)."""
    y = np.asarray(y, float); n = len(y)
    if win <= 1 or n == 0:
        return y.copy()
    half = win // 2
    csum = np.concatenate(([0.0], np.cumsum(y)))
    out  = np.empty(n)
    for k in range(n):
        lo, hi = max(0, k - half), min(n, k + half + 1)
        out[k] = (csum[hi] - csum[lo]) / (hi - lo)
    return out

if has_piston_force:
    peak_i      = int(P_piston_ts.argmax())
    peak_P_step = int(steps_pf[peak_i])
    P_roll      = rolling_mean(P_piston_ts, roll_win)   # smoothed raw pressure

    fig, (axL, axG) = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    fig.suptitle(f'Piston pressure history:  {sim_name}\n'
                 f'$A = l_x\\,l_y = {piston_area:.1f}\\,\\sigma^2$',
                 fontsize=14, fontweight='bold')

    for ax in (axL, axG):
        ax.axvline(peak_P_step, color=WONG['blue'], ls='--', lw=1.6, alpha=0.7,
                   label=f'peak $P$ / relaxation start (t={peak_P_step})')
        ax.set_xlabel('step'); ax.grid(alpha=0.3)

    # (a) linear -----------------------------------------------------------
    axL.plot(steps_pf, P_piston_ts, '-', color=WONG['vermillion'], lw=1.0, alpha=0.30,
             label=r'$P = F_z/A$ (raw)')
    axL.plot(steps_pf, P_roll, '-', color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label=f'rolling mean ({roll_win} pts)')
    if has_piston_force_avg:
        axL.plot(steps_pf_avg, P_piston_avg_ts, '-', color=WONG['black'], lw=1.8,
                 alpha=0.8, label='LMP block-avg')
    axL.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    axL.set_ylabel(r'$P = F_z/A$  (LJ / $\sigma^2$)')
    axL.set_title('(a) piston pressure')
    axL.legend(fontsize=13, loc='best')

    # (b) log  -- guard non-positive values (occur before the piston contacts gel)
    _pos = P_piston_ts > 0
    axG.plot(steps_pf[_pos], np.log(P_piston_ts[_pos]), '-',
             color=WONG['vermillion'], lw=1.0, alpha=0.30, label=r'$\ln(F_z/A)$ (raw)')
    _posr = P_roll > 0
    axG.plot(steps_pf[_posr], np.log(P_roll[_posr]), '-',
             color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label=f'rolling mean ({roll_win} pts)')
    if has_piston_force_avg:
        _posa = P_piston_avg_ts > 0
        axG.plot(steps_pf_avg[_posa], np.log(P_piston_avg_ts[_posa]), '-',
                 color=WONG['black'], lw=1.8, alpha=0.8, label='LMP block-avg')
    axG.set_ylabel(r'$\ln P$')
    axG.set_title('(b) log piston pressure (relaxation view)')
    axG.legend(fontsize=13, loc='best')

    out_P = PLOT_DIR / f'piston_pressure_history_{sim_name}.png'
    plt.savefig(out_P, dpi=150, bbox_inches='tight'); print('saved', out_P); plt.show()

    print(f'  peak  P = {P_piston_ts.max():.4f} LJ/sigma^2  at step {peak_P_step}')
    print(f'  final P (raw)         = {P_piston_ts[-1]:.4f} LJ/sigma^2  at step {steps_pf[-1]}')
    print(f'  final P (rolling {roll_win}) = {P_roll[-1]:.4f} LJ/sigma^2')
else:
    print('skipped (no piston_force file)')


## 4b — Piston force history (relaxation check)

$F_{z,\mathrm{piston}}(t)$: the raw summed pairwise z-force on the piston atoms
from `fix out_piston_force` in `triaxial_compression.lmp` (same series that
divided by $A=l_xl_y$ gives the pressure above).  Same format as the piston
pressure panels: raw + notebook rolling mean (+ LMP block-avg if present), peak
marked, with a **log panel** so the relaxation plateau is easy to read once
$\ln F_z$ flattens.

In [ ]:
# ---- piston force F_z vs timestep: linear + log ----
# Same format as the piston-pressure cell above, but plotting the raw force
# F_z (not F_z/A).  Reuses roll_win and rolling_mean() defined in the pressure
# cell.  Raw c_piston_fz is thermally noisy (no piston damping, by design), so
# we reduce visual noise by TIME-AVERAGING: a centered rolling mean of the raw
# series, and the LMP block-averaged series if the run has it.
if has_piston_force:
    peakF_i      = int(F_piston.argmax())
    peakF_step   = int(steps_pf[peakF_i])
    F_roll       = rolling_mean(F_piston, roll_win)      # smoothed raw force

    fig, (axL, axG) = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    fig.suptitle(f'Piston force history:  {sim_name}\n'
                 f'$F_z$ = summed pairwise z-force on piston atoms',
                 fontsize=14, fontweight='bold')

    for ax in (axL, axG):
        ax.axvline(peakF_step, color=WONG['blue'], ls='--', lw=1.6, alpha=0.7,
                   label=f'peak $F_z$ / relaxation start (t={peakF_step})')
        ax.set_xlabel('step'); ax.grid(alpha=0.3)

    # (a) linear -----------------------------------------------------------
    axL.plot(steps_pf, F_piston, '-', color=WONG['vermillion'], lw=1.0, alpha=0.30,
             label=r'$F_z$ (raw)')
    axL.plot(steps_pf, F_roll, '-', color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label=f'rolling mean ({roll_win} pts)')
    if has_piston_force_avg:
        axL.plot(steps_pf_avg, F_piston_avg, '-', color=WONG['black'], lw=1.8,
                 alpha=0.8, label='LMP block-avg')
    axL.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    axL.set_ylabel(r'$F_z$  (LJ units)')
    axL.set_title('(a) piston force')
    axL.legend(fontsize=13, loc='best')

    # (b) log  -- guard non-positive values (occur before the piston contacts gel)
    _pos = F_piston > 0
    axG.plot(steps_pf[_pos], np.log(F_piston[_pos]), '-',
             color=WONG['vermillion'], lw=1.0, alpha=0.30, label=r'$\ln F_z$ (raw)')
    _posr = F_roll > 0
    axG.plot(steps_pf[_posr], np.log(F_roll[_posr]), '-',
             color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label=f'rolling mean ({roll_win} pts)')
    if has_piston_force_avg:
        _posa = F_piston_avg > 0
        axG.plot(steps_pf_avg[_posa], np.log(F_piston_avg[_posa]), '-',
                 color=WONG['black'], lw=1.8, alpha=0.8, label='LMP block-avg')
    axG.set_ylabel(r'$\ln F_z$')
    axG.set_title('(b) log piston force (relaxation view)')
    axG.legend(fontsize=13, loc='best')

    out_F = PLOT_DIR / f'piston_force_history_{sim_name}.png'
    plt.savefig(out_F, dpi=150, bbox_inches='tight'); print('saved', out_F); plt.show()

    print(f'  peak  F_z = {F_piston.max():.2f} LJ  at step {peakF_step}')
    print(f'  final F_z (raw)         = {F_piston[-1]:.2f} LJ  at step {steps_pf[-1]}')
    print(f'  final F_z (rolling {roll_win}) = {F_roll[-1]:.2f} LJ')
else:
    print('skipped (no piston_force file)')


## 5 — Longitudinal modulus $M$: network stress vs piston pressure

Two independent estimates of the drained longitudinal modulus at the final
(relaxed) state, strain $\varepsilon = $ `comp_percent` ($\approx 10\%$):

- **Network:** $M_\mathrm{net} = \langle\sigma'_{zz}\rangle_\mathrm{membrane}/\varepsilon$
  — the network (effective) stress from §3 averaged over the membrane bins,
  divided by strain.  Error bar = 95 % CI from the bin-to-bin spatial scatter.
- **Piston:** $M_\mathrm{piston} = (F_z/A)/\varepsilon$ — the relaxed piston
  pressure divided by strain.  Error bar = piston-force fluctuation over the
  final averaging window propagated through $1/(A\varepsilon)$.

Agreement validates the Terzaghi network-stress decomposition. (Same comparison
as in `compression_analysis.ipynb`.)

In [ ]:
# ==========================================================================
#  5 — LONGITUDINAL MODULUS M:  network stress  vs  piston pressure
# ==========================================================================
# Strain actually reached (nominal target = comp_percent = 10%).
eps_final = float(strain_eps[-1])
print(f'strain: eps_final = {eps_final:.4f}   (nominal comp_percent = {comp_percent})')

# ---- Method 1: network (effective) stress  M = <sigma'_zz>_membrane / eps ----
# net_stack[-1] and in_mem come from section 3 (cell "3 - NETWORK ... STRESS").
Mnet_bins = net_stack[-1][in_mem] / eps_final          # per-bin M in the membrane
Mnet_bins = Mnet_bins[np.isfinite(Mnet_bins)]
M_net, M_net_lo, M_net_hi = mean_ci(Mnet_bins, ci_level)
ci_pct = int(ci_level * 100)
print(f'M_network = {M_net:.4f}  [{M_net_lo:.4f}, {M_net_hi:.4f}]  '
      f'({ci_pct}% CI, {len(Mnet_bins)} membrane bins)')

# ---- Method 2: piston pressure  M = (F_z/A) / eps ----
if has_piston_force:
    # Final plateau window = last coarse-stress epoch [prod_ts[-1]-nfreq, prod_ts[-1]],
    # matching the window that produced the final network-stress snapshot.
    nfreq_est = int(prod_ts[-1] - prod_ts[-2]) if len(prod_ts) >= 2 else int(prod_ts[-1])
    win_lo, win_hi = prod_ts[-1] - nfreq_est, prod_ts[-1]
    _mw   = (steps_pf >= win_lo) & (steps_pf <= win_hi)
    F_win = F_piston[_mw]
    if len(F_win) < 2:                                 # fall back to the last few samples
        F_win = F_piston[-5:]
    F_mean = float(np.mean(F_win))
    F_sem  = float(stats.sem(F_win)) if len(F_win) > 1 else 0.0
    P_piston_final = F_mean / piston_area
    M_piston       = P_piston_final / eps_final
    M_piston_err   = F_sem / (piston_area * eps_final)  # naive SE (ignores autocorr.)
    print(f'M_piston  = {M_piston:.4f} +/- {M_piston_err:.4f}   '
          f'(window {win_lo}-{win_hi}, {len(F_win)} samples, '
          f'P={P_piston_final:.4f} LJ/sigma^2)')
    print(f'ratio M_piston / M_network = {M_piston / M_net:.4f}')

# ---- comparison plot with error bars ----
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
ax.errorbar([0], [M_net], yerr=[[M_net - M_net_lo], [M_net_hi - M_net]],
            fmt='o', ms=13, color=WONG['blue'], capsize=8, lw=2.5,
            label=(f"network $\\sigma^{{\\prime}}_{{zz}}$   $M = {M_net:.3f}$\n"
                   f"{ci_pct}% CI [{M_net_lo:.3f}, {M_net_hi:.3f}]"))
if has_piston_force:
    ax.errorbar([1], [M_piston], yerr=[[M_piston_err], [M_piston_err]],
                fmt='s', ms=13, color=WONG['vermillion'], capsize=8, lw=2.5,
                label=f"piston $F_z/A$   $M = {M_piston:.3f} \\pm {M_piston_err:.3f}$")
    ax.axhline(M_piston, color=WONG['vermillion'], ls='--', lw=1.2, alpha=0.5)
ax.axhline(M_net, color=WONG['blue'], ls='--', lw=1.2, alpha=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels([r"network ($\sigma'/\epsilon$)", r'piston ($P/\epsilon$)'], fontsize=17)
ax.set_ylabel(r'$M$  (LJ units)')
ax.set_title(f'Longitudinal modulus comparison\n{RUN_ID}  |  '
             f'$\\varepsilon_{{zz}} = {eps_final:.3f}$')
ax.set_xlim(-0.5, 1.5); ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=14, loc='best')

out_M = PLOT_DIR / f'M_comparison_{sim_name}.png'
plt.savefig(out_M, dpi=150, bbox_inches='tight'); print('saved', out_M); plt.show()


## 6 — Stress-strain sweep summary: piston pressure vs strain
One point per compression level (`_c<level>`). $P=\langle F_z\rangle/A$ from the block-averaged
piston force at the end of each hold; $\varepsilon$ from `strain_zz`. The initial secant slope
$dP/d\varepsilon$ is the small-deformation longitudinal modulus $M$. Set `COMP_LEVELS` to match
`COMPRESSIONS` in `triaxial_compression.batch`, and stage each level (sync cell) first.

In [ ]:
# Stress-strain sweep: P_piston vs gel strain across compression levels.
import numpy as np, matplotlib.pyplot as plt

# COMP_LEVELS comes from the Config cell (the single source of truth for the
# sweep).  Edit it there, not here.
N_TAIL = 5   # trailing block rows averaged for the equilibrium (relaxed) value

def _load2(path):
    rows = []
    for ln in open(path):
        ln = ln.strip()
        if not ln or ln.startswith('#'):
            continue
        try:
            rows.append([float(x) for x in ln.split()])
        except ValueError:
            continue
    return np.array(rows)

eps_list, P_list = [], []
for lvl in COMP_LEVELS:
    base = f'{DATANAME}_{INTERACTION}_{NSTEPS}_c{lvl}'
    fF = DATA_DIR / f'piston_force_avg_{base}.dat'
    fS = DATA_DIR / f'strain_zz_{base}.dat'
    fB = DATA_DIR / f'box_dimensions_{base}.dat'
    if not (fF.exists() and fS.exists() and fB.exists()):
        print(f'  level {lvl}: missing files, skipping')
        continue
    F, S, B = _load2(fF), _load2(fS), _load2(fB)
    Fz = F[-N_TAIL:, 1].mean()
    Linit, Lcur = S[-N_TAIL:, 1].mean(), S[-N_TAIL:, 2].mean()
    eps = (Linit - Lcur) / Linit
    A = (B[-N_TAIL:, 1] * B[-N_TAIL:, 2]).mean()      # lx * ly
    P = Fz / A
    eps_list.append(eps); P_list.append(P)
    print(f'  level {lvl}: eps={eps:.4f}  <F_z>={Fz:.4g}  A={A:.4g}  P={P:.4g}')

eps_arr, P_arr = np.array(eps_list), np.array(P_list)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(eps_arr, P_arr, 'o-', lw=2)
if len(eps_arr) >= 2:
    M_init = (P_arr[1] - P_arr[0]) / (eps_arr[1] - eps_arr[0])
    M_sec  = (P_arr[-1] - P_arr[0]) / (eps_arr[-1] - eps_arr[0])
    ax.set_title(f'Piston pressure vs strain   |   M_init \u2248 {M_init:.3g}   M_secant \u2248 {M_sec:.3g}')
ax.set_xlabel('gel strain  \u03b5'); ax.set_ylabel('piston pressure  P = <F_z> / A  (LJ)')
ax.grid(alpha=0.3)
out = PLOT_DIR / f'stress_strain_sweep_{DATANAME}_{INTERACTION}_{NSTEPS}.png'
fig.tight_layout(); fig.savefig(out, dpi=150); print('saved', out); plt.show()